# 03 · Network agents (the supermaster)
The supermaster runs eight agents that see the whole network:

| Agent | Job | Changes lights? |
|---|---|---|
| Demand forecast | traffic level and direction 30 min ahead | no (informs) |
| Incident detection | blocked lanes and congestion, never names a cause | no (publishes facts) |
| Rerouting | detour advice, trip planner | no (advice only) |
| Emergency route | warns junctions ahead of emergency vehicles | indirectly (directive) |
| Corridor | green waves between neighbouring junctions | indirectly (directive) |
| Transit monitor | which buses are late | indirectly (feeds transit priority) |
| Weather | rain and floods → longer timings | indirectly (directive) |
| Notification | alerts for the seven persona pages | no |

We run 25 minutes of the **combined** scenario: rain from minute 5, a blocked lane on Bourguiba from
minute 15, and ambulances.

In [ ]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath(".."))          # the project root, so `traffix` can be imported
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option("display.max_colwidth", 160); pd.set_option("display.width", 200)
from traffix.core import config
from traffix.evaluation import notebook_tools as nt

In [ ]:
def capture(now, system):
    pubs = system.bus.publications
    kinds = pd.Series([k for k, _ in pubs]).value_counts().to_dict()
    return {"minute": now / 60, "rain": system.sensors.weather.get("rain", 0),
            "forecast level": system.bus.forecast.get("level"), "incidents": kinds.get("INCIDENT", 0),
            "congested roads": kinds.get("CONGESTION", 0), "late buses": kinds.get("TRANSIT_LATE", 0),
            "notifications": len(system.bus.notifications)}
result, system, snaps, narr = nt.short_run("combined", minutes=25, every=30, capture=capture)
ts = pd.DataFrame(snaps).set_index("minute")
ts.tail()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
ts[["rain"]].plot(ax=axes[0], title="Rain reported by the weather feed")
ts[["incidents", "congested roads", "late buses"]].plot(ax=axes[1], title="Facts published by network agents")
ts[["notifications"]].plot(ax=axes[2], title="Notifications sent (all personas)")
for a in axes: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 1. What each network agent decided

In [ ]:
net_reasons = ["WEATHER", "INCIDENT", "CONGESTION", "REROUTE", "EMERGENCY_PREPARE", "CORRIDOR", "FORECAST"]
nt.decisions(system, narr, reasons=net_reasons).groupby("reason").size().rename("decisions").to_frame()

In [ ]:
for reason in ["WEATHER_RAIN", "INCIDENT_SUSPECTED", "REROUTE", "EMERGENCY_PREPARE", "FORECAST_UPDATE"]:
    d = nt.decisions(system, narr, reasons=[reason], limit=1)
    if len(d): print(f"{reason}:\n   {d.iloc[0]['time']} · {d.iloc[0]['what happened']}\n")

## 2. Incident detection quality
The blocked lane starts at minute 15. How fast was it detected, and were there false alarms?

In [ ]:
print("Detected:", bool(result.get("incident_detected")), "· delay:", result.get("incident_detect_delay_s"), "s",
      "· false alarms:", result.get("incident_false_alarms"))

## 3. Notifications for each persona, in three languages

In [ ]:
notes = pd.DataFrame(system.bus.notifications)
display(notes.groupby(["persona", "type"]).size().unstack(fill_value=0))
for t in ["INCIDENT", "AMBULANCE_NEAR", "RAIN"]:
    n = next((x for x in system.bus.notifications if x["type"] == t), None)
    if n:
        print(f"\n{t} (for {n['persona']}):")
        for lang in ("en", "fr", "ar"): print(f"  {lang}: {n['texts'][lang]}")